In [1]:
import os

In [2]:
%pwd

'c:\\Users\\lenov\\OneDrive\\Pictures\\Kidney-disease-classification-Deep-learning-project\\research'

In [3]:
os.chdir("../")

In [4]:
%pwd

'c:\\Users\\lenov\\OneDrive\\Pictures\\Kidney-disease-classification-Deep-learning-project'

In [5]:
from dataclasses import dataclass

from pathlib import Path





@dataclass(frozen=True)

class TrainingConfig:

    root_dir: Path

    trained_model_path: Path

    updated_base_model_path: Path

    training_data: Path

    params_epochs: int

    params_batch_size: int

    params_is_augmentation: bool

    params_image_size: list

    params_learning_rate: float

In [6]:
from cnnClassifier.constants import *

from cnnClassifier.utils.common import read_yaml, create_directories

import tensorflow as tf

In [7]:
class ConfigurationManager:

    def __init__(

        self,

        config_filepath = CONFIG_FILE_PATH,

        params_filepath = PARAMS_FILE_PATH):



        self.config = read_yaml(config_filepath)

        self.params = read_yaml(params_filepath)



        create_directories([self.config.artifacts_root])





    

    def get_training_config(self) -> TrainingConfig:

        training = self.config.training

        prepare_base_model = self.config.prepare_base_model

        params = self.params

        training_data = os.path.join(self.config.data_ingestion.unzip_dir, "kidney-ct-scan-image")

        create_directories([

            Path(training.root_dir)

        ])



        training_config = TrainingConfig(

            root_dir=Path(training.root_dir),

            trained_model_path=Path(training.trained_model_path),

            updated_base_model_path=Path(prepare_base_model.updated_base_model_path),

            training_data=Path(training_data),

            params_epochs=params.EPOCHS,

            params_batch_size=params.BATCH_SIZE,

            params_is_augmentation=params.AUGMENTATION,

            params_image_size=params.IMAGE_SIZE,

            params_learning_rate=params.LEARNING_RATE

        )



        return training_config

In [8]:
import os

import urllib.request as request

from zipfile import ZipFile

import tensorflow as tf

import time

from xml.parsers.expat import model

In [9]:
class Training:

    def __init__(self, config: TrainingConfig):

        self.config = config



    

    def get_base_model(self):

        self.model = tf.keras.models.load_model(

            self.config.updated_base_model_path

        )

        self.model.compile(

            optimizer=tf.keras.optimizers.Adam(learning_rate=self.config.params_learning_rate),

            loss=tf.keras.losses.CategoricalCrossentropy(),

            metrics=["accuracy", tf.keras.metrics.Precision(), tf.keras.metrics.Recall()]

        )



    def train_valid_generator(self):



        datagenerator_kwargs = dict(

            rescale = 1./255,

            validation_split=0.20

        )



        dataflow_kwargs = dict(

            target_size=self.config.params_image_size[:-1],

            batch_size=self.config.params_batch_size,

            interpolation="bilinear"

        )



        valid_datagenerator = tf.keras.preprocessing.image.ImageDataGenerator(

            **datagenerator_kwargs

        )



        self.valid_generator = valid_datagenerator.flow_from_directory(

            directory=self.config.training_data,

            subset="validation",

            shuffle=False,

            **dataflow_kwargs

        )



        if self.config.params_is_augmentation:

            train_datagenerator = tf.keras.preprocessing.image.ImageDataGenerator(

                rotation_range=40,

                horizontal_flip=True,

                width_shift_range=0.2,

                height_shift_range=0.2,

                shear_range=0.2,

                zoom_range=0.2,

                **datagenerator_kwargs

            )

        else:

            train_datagenerator = valid_datagenerator



        self.train_generator = train_datagenerator.flow_from_directory(

            directory=self.config.training_data,

            subset="training",

            shuffle=True,

            **dataflow_kwargs

        )



    

    @staticmethod

    def save_model(path: Path, model: tf.keras.Model):

        model.save(path)



    

    def train(self):

        self.steps_per_epoch = self.train_generator.samples // self.train_generator.batch_size

        self.validation_steps = self.valid_generator.samples // self.valid_generator.batch_size



    



        self.model.fit(

            self.train_generator,

            epochs=self.config.params_epochs,

            steps_per_epoch=self.steps_per_epoch,

            validation_steps=self.validation_steps,

            validation_data=self.valid_generator

        )



        self.save_model(

            path=self.config.trained_model_path,

            model=self.model

        )

In [1]:
import tensorflow as tf
print(tf.config.list_physical_devices())



[PhysicalDevice(name='/physical_device:CPU:0', device_type='CPU'), PhysicalDevice(name='/physical_device:GPU:0', device_type='GPU')]


In [10]:
try:

    config = ConfigurationManager()

    training_config = config.get_training_config()

    training = Training(config=training_config)

    training.get_base_model()

    training.train_valid_generator()

    training.train()

    

except Exception as e:

    raise e

[2026-05-01 15:18:07,269: INFO: common: yaml file: config\config.yaml loaded successfully]
[2026-05-01 15:18:07,269: INFO: common: yaml file: params.yaml loaded successfully]
[2026-05-01 15:18:07,269: INFO: common: created directory at: artifacts]
[2026-05-01 15:18:07,269: INFO: common: created directory at: artifacts\training]
[2026-05-01 15:18:07,463: WARNING: config: TensorFlow GPU support is not available on native Windows for TensorFlow >= 2.11. Even if CUDA/cuDNN are installed, GPU will not be used. Please use WSL2 or the TensorFlow-DirectML plugin.]
[2026-05-01 15:18:07,463: WARNING: saving_utils: Compiled the loaded model, but the compiled metrics have yet to be built. `model.compile_metrics` will be empty until you train or evaluate the model.]
Found 382 images belonging to 4 classes.
Found 1531 images belonging to 4 classes.
Epoch 1/5
95/95 ━━━━━━━━━━━━━━━━━━━━ 82s 860ms/step - accuracy: 0.6277 - loss: 5.5304 - precision: 0.6313 - recall: 0.6238 - val_accuracy: 0.2283 - val_l

c:\Users\lenov\OneDrive\Pictures\Kidney-disease-classification-Deep-learning-project\.venv\lib\site-packages\keras\src\trainers\epoch_iterator.py:116: UserWarning: Your input ran out of data; interrupting training. Make sure that your dataset or generator can generate at least `steps_per_epoch * epochs` batches. You may need to use the `.repeat()` function when building your dataset.
  self._interrupted_warning()


95/95 ━━━━━━━━━━━━━━━━━━━━ 17s 177ms/step - accuracy: 0.5625 - loss: 4.3943 - precision: 0.5625 - recall: 0.5625 - val_accuracy: 0.4484 - val_loss: 10.6321 - val_precision: 0.4484 - val_recall: 0.4484
Epoch 3/5
95/95 ━━━━━━━━━━━━━━━━━━━━ 82s 868ms/step - accuracy: 0.8356 - loss: 1.6924 - precision: 0.8361 - recall: 0.8350 - val_accuracy: 0.5082 - val_loss: 10.8449 - val_precision: 0.5082 - val_recall: 0.5082
Epoch 4/5
95/95 ━━━━━━━━━━━━━━━━━━━━ 17s 172ms/step - accuracy: 1.0000 - loss: 1.3389e-04 - precision: 1.0000 - recall: 1.0000 - val_accuracy: 0.5245 - val_loss: 10.8375 - val_precision: 0.5245 - val_recall: 0.5245
Epoch 5/5
95/95 ━━━━━━━━━━━━━━━━━━━━ 83s 877ms/step - accuracy: 0.8660 - loss: 1.3456 - precision: 0.8672 - recall: 0.8660 - val_accuracy: 0.2717 - val_loss: 16.1411 - val_precision: 0.2717 - val_recall: 0.2717
[2026-05-01 15:22:53,536: WARNING: saving_api: You are saving your model as an HDF5 file via `model.save()` or `keras.saving.save_model(model)`. This file format 